# 1.SummarizationMiddleware中间件

## 举例1：测试trigger、keep参数

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

summary_model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
    profile = {"max_input_tokens":64000}   # 务必显式赋值
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,AIMessage,SystemMessage

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]


agent = create_agent(
    model = model,
    middleware = [
        SummarizationMiddleware(
            model = summary_model,
            trigger = [
                ("tokens",100),
                ("messages", 6),
                ("fraction", 0.001)   # max_input_tokens * fraction
            ],
            keep =("messages", 2)
        )
    ]
)

res = agent.invoke({
    "messages":messages
})

for msg in res["messages"]:
    msg.pretty_print()

## 举例2：summary_prompt的使用

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,AIMessage,SystemMessage

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]


agent = create_agent(
    model = model,
    middleware = [
        SummarizationMiddleware(
            model = summary_model,
            trigger = [
                ("tokens",100),
                ("messages", 6),
                ("fraction", 0.001)   # max_input_tokens * fraction
            ],
            keep =("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"   # 该提示词需要包含{messages}占位符，使得历史消息列表可以被插入
        )
    ]
)

res = agent.invoke({
    "messages":messages
})

for msg in res["messages"]:
    msg.pretty_print()